In [2]:
# -*- coding: utf-8 -*-
"""
Created on Wed Jul 15 10:08:31 2020

@author: PristerM
"""
'''
Updated Aug 18 2025
JyW
'''
# %%

#------------------------------------------------ Begin_Librairie ----------------------------------------

from selenium import webdriver
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import pandas as pd
import datetime
from time import sleep
import os
import re
from pandas import ExcelWriter



In [3]:
#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'BI BRB' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.1.2")

now=datetime.datetime.now()

filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"
#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

writer = ExcelWriter(filename, engine='openpyxl')

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process



if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)


Running BI BRB Web Scraping Tool v.1.2


In [4]:

# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder,

		 'profile.default_content_setting_values.automatic_downloads': 1}

chromeOptions.add_experimental_option("prefs",prefs)


driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()



In [184]:
# %%

#------------------------------------------------ Begin_Variable ----------------------------------------

regdict={
    regulatorName + ' 1': 'https://www.brb.bi/node/119', 

	regulatorName + ' 2': 'https://www.brb.bi/node/120', 

	 regulatorName + ' 3': 'https://www.brb.bi/node/1551', 
   }

Typology={

		regulatorName + ' 1': 'List of "Banking Supervision"',
		regulatorName + ' 2': 'List of "Microfinance Supervision"',
		regulatorName + ' 3': 'List of "Payment Systems"',
        }

sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 

		  'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 

		  'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 

		  'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],

		  'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 

		  'Phone - Mother company': [], 'Check': []}


processdate=now.strftime('%Y-%m-%d')


In [179]:
# %%

#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict


def extract_first_phone(phone_text):
    match = re.search(r'\(?\+?\d{1,4}\)?[\s-]?\d{2,4}[\s-]?\d{2,4}[\s-]?\d{2,4}', phone_text)
    return match.group(0) if match else ''


In [185]:
# %%

#------------------------------------------------ Begin_Main ----------------------------------------
for reg in regdict:

    print(f'Working with list {reg}')
    driver.get(regdict[reg])
    sleep(3)
    soup = BeautifulSoup(driver.page_source, 'html.parser')  
    sleep(5)
    if reg == regulatorName + ' 1':
        current_company = {}
        skip_titles = ["A.", "B."]  # Titles to skip

        # Iterate through all <p> tags
        for p in soup.find_all('p'):
            strong_tag = p.find('strong')
            if strong_tag:
                title = strong_tag.get_text(strip=True)
                # Check if the title starts with a skip prefix
                if any(title.startswith(prefix) for prefix in skip_titles) or title[0].isdigit():
                    continue
                # Save the current company if it exists
                if current_company:
                    for key in sqldict.keys():
                        sqldict[key].append(current_company.get(key, ''))
                    current_company = {}
                # Start a new company
                current_company['Name'] = title
                current_company['ListProcessDate'] = processdate
                current_company['ListName'] = Typology[reg]
                current_company['ListCode'] = reg.split(' ')[-1]
                current_company['RegCtry'] = reg.split(' ')[0]
                current_company['RegCode'] = reg.split(' ')[1]
                current_company['RegulationType'] = 'Regulated'
            else:
                text = p.get_text(strip=True)
                if "CAPITAL SOCIAL" in text:
                    current_company['InternalID_1'] = text.split(':')[-1].strip().rstrip()
                    current_company['InternalID_1_type'] = 'CAPITAL SOCIAL'
                elif 'Swift' in text:
                    current_company['BIC SWIFT Code'] = text.split(':')[-1].strip().rstrip()
                elif 'NIF' in text:
                    current_company['InternalID_2'] = text.split(':')[-1].strip().rstrip()
                    current_company['InternalID_2_type'] = 'NIF'
                elif "Tél" in text:
                    match = re.search(r'Tél[:\s]*(.*?)(,|$)', text)
                    if match:
                        current_company['Phone'] = match.group(1).strip()
                elif "Courriel" in text:
                    email_tag = p.find('a')
                    if email_tag:
                        current_company['Email'] = email_tag.get_text(strip=True)
                elif "Web" in text or "Site Web" in text:
                    website_tag = p.find('a')
                    if website_tag:
                        current_company['Website'] = website_tag.get_text(strip=True)
                elif "BP" in text:
                    current_company['Address_1'] = text

        # Append the last company
        if current_company:
            for key in sqldict.keys():
                sqldict[key].append(current_company.get(key, ''))

        # Align all fields in `sqldict`
        sqldict = bourange_same_length_array(sqldict)
    if reg == regulatorName + ' 2':
        infos = soup.find_all('p')
        other_info = ''
        phone = ''
        regulation_date = ''
        top = ''
        name = ''
        each_index = -1  # Track the last valid index

        for index, info in enumerate(infos):
            if index < 4:
                continue

            text = info.get_text(strip=True)

            if 'Institutions de microfinance de quatrième catégorie' in info.text:
                break

            if not text:
                continue

            # If a new entry starts (e.g., numbered list), process the previous `other_info` and reset
            if re.match(r'^\d+\.\s*', text):
                # Append all fields for the previous entry
                if other_info or name:  # Ensure alignment only if there is valid data
                    sqldict['Address_1'].append(other_info.strip() if other_info else '')
                    sqldict['Name'].append(name if name else '')
                    sqldict['Phone'].append(phone if phone else '')
                    sqldict['RegulationDate'].append(regulation_date if regulation_date else '')
                    sqldict['Typology'].append(top if top else '')
                    sqldict['ListProcessDate'].append(processdate)

                    sqldict['ListName'].append(Typology[reg])
                    sqldict['ListCode'].append(reg.split(' ')[-1])
                    sqldict['RegCtry'].append(reg.split(' ')[0])
                    sqldict['RegCode'].append(reg.split(' ')[1])
                    sqldict['RegulationType'].append('Regulated')

                # Reset for the next entry
                other_info = ''
                phone = ''
                regulation_date = ''
                top = ''
                each_index = index
                name = re.sub(r'^\d+\.\s*', '', text)  # Extract the name
                #print(name)

            elif 'Tél' in text or 'Tel' in text:
                if 'Tél' in text:
                    phone = text[text.find('Tél') + 4:].strip()
                elif 'Tel' in text:
                    phone = text[text.find('Tel') + 4:].strip()

            elif 'Date' in text:
                regulation_date = text.split(':')[-1].strip()

            elif 'Société' in text:
                top = text

            else:
                # Accumulate `other_info` for the current entry
                if index > each_index:
                    other_info += text + ' '

    # Append the last entry after the loop ends
        if other_info or name:
            sqldict['Address_1'].append(other_info.strip() if other_info else '')
            sqldict['Name'].append(name if name else '')
            sqldict['Phone'].append(phone if phone else '')
            sqldict['RegulationDate'].append(regulation_date if regulation_date else '')
            sqldict['Typology'].append(top if top else '')
            sqldict['ListProcessDate'].append(processdate)


            sqldict['ListName'].append(Typology[reg])
            sqldict['ListCode'].append(reg.split(' ')[-1])
            sqldict['RegCtry'].append(reg.split(' ')[0])
            sqldict['RegCode'].append(reg.split(' ')[1])
            sqldict['RegulationType'].append('Regulated')

        sqldict = bourange_same_length_array(sqldict)
        
    if reg == regulatorName + ' 3':
        table = soup.find('section',id='block-solo-content').find('table')
        table_rows = table.find_all('tr')
        for row in table_rows:
            cols = row.find_all('td')
            for index,col in enumerate(cols):
                strong_tag = col.find('strong')
                if strong_tag:
                # print(strong_tag.text)
                    pass
                elif col.text.strip():  # Check if the column text is not empty
                    #print(col.text)
                    if index == 0:
                        name = col.text.strip()
                        name = re.sub(r'^\d+\.\s*', '', name) 
                        #print(name)
                    elif index == 3:
                        phone = col.text.strip()
                        #print(phone)
                        #print(f"Name: {name}, Phone: {phone}")
                        sqldict['Name'].append(name)
                        sqldict['Phone'].append(phone)
                        sqldict['ListProcessDate'].append(processdate)
                        sqldict['ListName'].append(Typology[reg])
                        sqldict['ListCode'].append(reg.split(' ')[-1])
                        sqldict['RegCtry'].append(reg.split(' ')[0]) 
                        sqldict['RegCode'].append(reg.split(' ')[1])
                    sqldict = bourange_same_length_array(sqldict)


Working with list BI BRB 1
Working with list BI BRB 2
Working with list BI BRB 3


In [186]:
# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

os.chdir(scriptfolder)

df=pd.DataFrame(sqldict)

df['Zip'] = df['Address_1'].str.extract(r'(\b\d{4}\b)')[0]
df['Phone'] = df['Phone'].str.strip(':;,').str.rstrip(':;,')
df['Address_1'] = df['Address_1'].str.split('INSTITUTIONSÂ').str[0].str.strip()
# Example list of cities (you can expand this list based on your data)
cities=['Bubanza', 'Buhongo', 'Bujumbura', 'Bukirasazi', 'Bururi', 'Cankuzo', 'Cibitoke', 'Gitega', 'Kabezi', 'Karuzi', 'Kayanza', 'Kayero', 'Kayogoro', 'Kibondo', 'Kirundo', 'Kisozi', 'Luhwa', 'Makamba', 'Magara', 'Mukenke', 'Muramvya', 'Murore', 'Musenyi', 'Muyaga', 'Muyinga', 'Mwaro', 'Ngozi', 'Nyanza-Lac', 'Rugari', 'Rumonge', 'Rutana', 'Ruyigi', 'Zanandore']
       
# Extract city names from Address_1
df['City'] = df['Address_1'].apply(lambda x: next((city for city in cities if city.lower() in x.lower()), ''))
df['Phone'] = df['Phone'].apply(extract_first_phone)
df.to_excel(writer, 'SQL Ready', index=False)

writer.save()

writer.close()

driver.quit()

sleep(3)

C:\Users\wuj1\AppData\Local\Temp\4\ipykernel_34268\1677591148.py:18: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(writer, 'SQL Ready', index=False)


AttributeError: 'OpenpyxlWriter' object has no attribute 'save'

In [187]:

key_value_counts = {key: len(values) for key, values in sqldict.items()}

# Print the counts
for key, count in key_value_counts.items():
    print(f"Key '{key}' has {count} values.")

Key 'bvdid' has 109 values.
Key 'priority' has 109 values.
Key 'ListLabel' has 109 values.
Key 'Typology' has 109 values.
Key 'EntryType' has 109 values.
Key 'Name' has 109 values.
Key 'InternalID_1' has 109 values.
Key 'InternalID_1_type' has 109 values.
Key 'InternalID_2' has 109 values.
Key 'InternalID_2_type' has 109 values.
Key 'InternalID_3' has 109 values.
Key 'InternalID_3_type' has 109 values.
Key 'CoType' has 109 values.
Key 'License_Type' has 109 values.
Key 'Address_1' has 109 values.
Key 'Address_2' has 109 values.
Key 'City' has 109 values.
Key 'Zip' has 109 values.
Key 'Cntry' has 109 values.
Key 'Phone' has 109 values.
Key 'Fax' has 109 values.
Key 'Website' has 109 values.
Key 'Email' has 109 values.
Key 'RegulationType' has 109 values.
Key 'RegulationTypeCode' has 109 values.
Key 'RegulationDate' has 109 values.
Key 'CancellationDate' has 109 values.
Key 'RegCtry' has 109 values.
Key 'RegCode' has 109 values.
Key 'ListCode' has 109 values.
Key 'ListLanguage' has 109 v

In [188]:
df.to_csv('total_2.csv')

In [156]:
df['Phone']

0        +257 22 20 40
1        +257 22 26 52
2     (257) 22 22 1352
3                 None
4     (257) 22 24 3206
5       (257) 22 22 06
6       (257) 22 28 03
7     (257) 22 20 1111
8                 None
9                 None
10    (257) 22 27 7767
11      (257) 22 28 78
12                None
13      (257) 22 22 76
14      (257) 22 40 51
15      (257) 22 22 28
Name: Phone, dtype: object

In [66]:
df.to_csv('list1_total.csv')

In [144]:
df.to_csv('list_1.csv')